# DREAMwalk hierarchy file for the NetMedGPT KG

Builds the drug/disease hierarchy that DREAMwalk uses for its semantic similarity network (teleport), as in the DREAMwalk paper (Bang et al., Nat Commun 2023, Methods "Drug and disease ontologies"):
- **drugs**: ATC classification, ATC codes from DrugBank (`drugbank_full_database.xml`);
- **diseases**: MeSH disease hierarchy (as for their MSI KG). KG diseases (MONDO IDs) are mapped to MeSH through the MONDO cross-references (`mondo.obo`); MeSH supplementary concept records (mostly rare diseases, `supp2026.xml`) are placed under the descriptors they are mapped to; tree numbers come from `desc2026.xml` (trees C, diseases, and F03, mental disorders).

Format as in DREAMwalk's `demo/demo_hierarchy.csv`: `child,parent`; leaves are KG `node_index` values (the node names of the DREAMwalk network file); roots `drug` and `disease`. Drug branch: ATC code levels (`C04AC01 -> C04AC -> C04A -> C04 -> C -> drug`); disease branch: `<node> -> <descriptor>_dis -> tree numbers -> C04_dis -> C_dis -> disease`.

Output: `data/dreamwalk_inputs/hierarchy_full_kg.csv`. Run from `revision_scripts/` (the first cell changes to the repository root).

In [1]:
import os
os.chdir('..')

import xml.etree.ElementTree as ET
import pandas as pd

IN_DIR = "data/dreamwalk_inputs"
NODES = "data/KG/nodes.csv"
OUT = os.path.join(IN_DIR, "hierarchy_full_kg.csv")
DISEASE_TREES = ("C", "F03")    # MeSH branches used for diseases

In [2]:
# drugs: DrugBank ID -> ATC codes
NS = "{http://www.drugbank.ca}"
atc = {}
for _, el in ET.iterparse(os.path.join(IN_DIR, "drugbank_full_database.xml")):
    if el.tag == NS + "drug":
        codes = [c.get("code") for c in el.findall(f"{NS}atc-codes/{NS}atc-code")]
        if codes:
            for i in el.findall(NS + "drugbank-id"):
                atc[i.text] = codes
        el.clear()

# diseases: MONDO -> MeSH IDs
mondo_mesh, cur = {}, None
for line in open(os.path.join(IN_DIR, "mondo.obo"), encoding="utf8"):
    line = line.strip()
    if line == "[Term]":
        cur = None
    elif line.startswith("id: MONDO:"):
        cur = int(line[10:])
    elif cur is not None and line.startswith("xref: MESH:"):
        mondo_mesh.setdefault(cur, set()).add(line.split()[1][5:])

# MeSH descriptors -> tree numbers; supplementary records -> mapped descriptors
trees = {}
for _, el in ET.iterparse(os.path.join(IN_DIR, "desc2026.xml")):
    if el.tag == "DescriptorRecord":
        tn = [t.text for t in el.iter("TreeNumber") if t.text.startswith(DISEASE_TREES)]
        if tn:
            trees[el.findtext("DescriptorUI")] = tn
        el.clear()
supp = {}
for _, el in ET.iterparse(os.path.join(IN_DIR, "supp2026.xml")):
    if el.tag == "SupplementalRecord":
        supp[el.findtext("SupplementalRecordUI")] = [d.text.lstrip("*") for d in el.findall("HeadingMappedToList/HeadingMappedTo/DescriptorReferredTo/DescriptorUI")]
        el.clear()
print(f"DrugBank IDs with ATC: {len(atc)} | MONDO terms with MeSH: {len(mondo_mesh)} | "
      f"disease descriptors: {len(trees)} | supplementary records: {len(supp)}")

DrugBank IDs with ATC: 5398 | MONDO terms with MeSH: 8125 | disease descriptors: 5194 | supplementary records: 324046


In [3]:
nodes = pd.read_csv(NODES, usecols=["node_index", "node_id", "node_type"])
edges = set()


def add_atc(code):
    # C04AC01 -> C04AC -> C04A -> C04 -> C -> drug
    chain = [code[:n] for n in (7, 5, 4, 3, 1) if len(code) >= n]
    for child, parent in zip(chain, chain[1:] + ["drug"]):
        edges.add((child, parent))


# MeSH tree-number node; top-level numbers get '_dis' (C04 -> C04_dis) so they never collide with ATC codes
def tree_node(tn):
    return tn if "." in tn else tn + "_dis"


def add_tree_number(tn):
    # C04.557.470 -> C04.557 -> C04_dis -> C_dis -> disease
    parts = tn.split(".")
    chain = [".".join(parts[:k]) for k in range(len(parts), 1, -1)] + [parts[0] + "_dis", parts[0][0] + "_dis", "disease"]
    for child, parent in zip(chain, chain[1:]):
        edges.add((child, parent))


drug_leaves = dis_leaves = 0
for idx, nid, ntype in nodes.itertuples(index=False):
    leaf = str(idx)
    if ntype == "drug" and str(nid) in atc:
        for code in atc[str(nid)]:
            edges.add((leaf, code))
            add_atc(code)
        drug_leaves += 1
    elif ntype == "disease":
        descs = set()
        for x in str(nid).split("_"):
            if x.strip():
                for m in mondo_mesh.get(int(float(x)), ()):
                    descs |= {m} if m.startswith("D") else set(supp.get(m, []))
        descs = {d for d in descs if d in trees}
        for d in descs:
            edges.add((leaf, d + "_dis"))
            for tn in trees[d]:
                edges.add((d + "_dis", tree_node(tn)))
                add_tree_number(tn)
        dis_leaves += bool(descs)

n_drug, n_dis = (nodes.node_type == "drug").sum(), (nodes.node_type == "disease").sum()
print(f"drugs in hierarchy: {drug_leaves} of {n_drug} ({100 * drug_leaves / n_drug:.1f}%)")
print(f"diseases in hierarchy: {dis_leaves} of {n_dis} ({100 * dis_leaves / n_dis:.1f}%)")

hier = pd.DataFrame(sorted(edges), columns=["child", "parent"])
hier.to_csv(OUT, index=False)
print(f"{len(hier)} rows saved to {OUT}")

drugs in hierarchy: 3026 of 7957 (38.0%)
diseases in hierarchy: 6610 of 17080 (38.7%)
42289 rows saved to data/dreamwalk_inputs/hierarchy_full_kg.csv


In [4]:
# check: every leaf reaches the root 'drug' or 'disease' (required by DREAMwalk)
parents = hier.groupby("child")["parent"].apply(list).to_dict()


def roots(node, seen=()):
    if node in ("drug", "disease"):
        return {node}
    out = set()
    for p in parents.get(node, []):
        if p not in seen:
            out |= roots(p, seen + (node,))
    return out


leaves = set(nodes.node_index.astype(str)) & set(hier.child)
bad = [l for l in leaves if not roots(l)]
mixed = [l for l in leaves if len(roots(l)) > 1]
print(f"leaves: {len(leaves)} | without root: {len(bad)} | reaching both roots: {len(mixed)}")

leaves: 9636 | without root: 0 | reaching both roots: 0
